# Animacion de PSO

Este notebook genera una animacion del movimiento de las particulas de PSO sobre la funcion de Rosenbrock en 2D y guarda el resultado en la carpeta `datos`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation

from funciones_heuristicas import run_particle_swarm_optimization
from funciones_objetivo import rosenbrock

In [ ]:
DATOS_DIR = Path("datos")
DATOS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "objective_function": rosenbrock,
    "swarm_size": 35,
    "dimension": 2,
    "lower_bounds": -2.048,
    "upper_bounds": 2.048,
    "inertia_weight": 0.7,
    "cognitive_weight": 1.5,
    "social_weight": 1.5,
    "max_iterations": 80,
    "seed": 42,
    "x_limits": (-2.0, 2.0),
    "y_limits": (-1.0, 3.0),
    "grid_points": 250,
    "gif_name": "animacion_pso_rosenbrock_2d.gif",
}

In [ ]:
resultado = run_particle_swarm_optimization(
    objective_function=CONFIG["objective_function"],
    swarm_size=CONFIG["swarm_size"],
    dimension=CONFIG["dimension"],
    lower_bounds=CONFIG["lower_bounds"],
    upper_bounds=CONFIG["upper_bounds"],
    inertia_weight=CONFIG["inertia_weight"],
    cognitive_weight=CONFIG["cognitive_weight"],
    social_weight=CONFIG["social_weight"],
    max_iterations=CONFIG["max_iterations"],
    seed=CONFIG["seed"],
)

historial_posiciones = resultado["positions_history"]
historial_global = resultado["global_best_history"]

print("Mejor solucion:", resultado["best_solution"])
print("Mejor valor:", resultado["best_value"])
print("Iteraciones:", resultado["iterations"])
print("Evaluaciones:", resultado["evaluations"])

In [ ]:
x = np.linspace(*CONFIG["x_limits"], CONFIG["grid_points"])
y = np.linspace(*CONFIG["y_limits"], CONFIG["grid_points"])
X, Y = np.meshgrid(x, y)
Z = np.array([rosenbrock(np.array([x1, x2], dtype=float)) for x1, x2 in zip(X.ravel(), Y.ravel())])
Z = Z.reshape(X.shape)

plt.figure(figsize=(8, 6))
niveles = np.logspace(-1, 3.5, 25)
plt.contour(X, Y, Z, levels=niveles, norm="log", cmap="viridis")

for posiciones in historial_posiciones[:5]:
    plt.scatter(posiciones[:, 0], posiciones[:, 1], color="steelblue", s=18, alpha=0.25)

plt.plot(historial_global[:, 0], historial_global[:, 1], color="crimson", linewidth=2, label="Mejor global")
plt.scatter(historial_global[0, 0], historial_global[0, 1], color="black", label="Inicio mejor global")
plt.scatter(historial_global[-1, 0], historial_global[-1, 1], color="gold", edgecolor="black", label="Final mejor global")
plt.title("Resumen de trayectoria de PSO")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
niveles = np.logspace(-1, 3.5, 25)
ax.contour(X, Y, Z, levels=niveles, norm="log", cmap="viridis")
ax.set_title("PSO sobre Rosenbrock 2D")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_xlim(CONFIG["x_limits"])
ax.set_ylim(CONFIG["y_limits"])

particulas = ax.scatter([], [], color="steelblue", s=28, alpha=0.85, label="Particulas")
mejor_global_punto, = ax.plot([], [], marker="o", color="crimson", markersize=7, label="Mejor global")
mejor_global_trayectoria, = ax.plot([], [], color="crimson", linewidth=2, alpha=0.9)
texto = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
ax.legend(loc="upper right")

def init():
    particulas.set_offsets(np.empty((0, 2)))
    mejor_global_punto.set_data([], [])
    mejor_global_trayectoria.set_data([], [])
    texto.set_text("")
    return particulas, mejor_global_punto, mejor_global_trayectoria, texto

def update(frame):
    posiciones = historial_posiciones[frame]
    mejor_global = historial_global[frame]
    particulas.set_offsets(posiciones)
    mejor_global_punto.set_data([mejor_global[0]], [mejor_global[1]])
    mejor_global_trayectoria.set_data(historial_global[: frame + 1, 0], historial_global[: frame + 1, 1])
    texto.set_text(f"Iteracion: {frame} | Mejor valor: {resultado['best_values_history'][frame]:.6f}")
    return particulas, mejor_global_punto, mejor_global_trayectoria, texto

animacion = animation.FuncAnimation(
    fig,
    update,
    frames=len(historial_posiciones),
    init_func=init,
    interval=120,
    blit=True,
)

plt.close(fig)
HTML(animacion.to_jshtml())

In [ ]:
gif_path = DATOS_DIR / CONFIG["gif_name"]
animacion.save(gif_path, writer=animation.PillowWriter(fps=10))
print(f"Animacion guardada en: {gif_path.resolve()}")